# Zif268 coronal heatmaps on the real Allen Mouse Brain Atlas

Run this notebook **locally** (not in a hosted/sandboxed environment) since the first cell
that touches `brainglobe_atlasapi` downloads the Allen Mouse Brain Atlas (a few hundred MB).
That download needs a normal internet connection; it's cached afterward in `~/.brainglobe`
so you only pay that cost once.

**What this notebook does, step by step:**
1. Install/import what's needed.
2. Load your raw IHC data (`IHC_effect_size_analysis.xlsx`, sheet `Raw IHC extracted`).
3. Run a one-way ANOVA (across SFPT / ST / FT) + Tukey HSD post-hoc, separately for each
   Region x Treatment group, and summarize the result as compact letter display (CLD)
   letters — no condition is treated as a reference/control.
4. Draw 3 figures (one per condition), each with 3 coronal sections × your treatment groups,
   colored by absolute mean % Zif268+ cells, with the CLD letters overlaid.

Edit the **Settings** cell below to match your file paths, regions, and groups, then
Run All.

## 1. Install packages
Run once. Comment out after the first successful run if you like.

In [196]:
%pip install -q "zarr>=3" brainglobe-heatmap brainglobe-atlasapi pandas matplotlib openpyxl scipy statsmodels
# brainglobe-atlasapi's newer atlas files are stored in zarr v3 format. If your environment
# already has an older zarr (v2.x) pulled in by something else, atlas loading fails with
# `GroupNotFoundError: group not found at path ''` even though the download succeeded --
# see https://github.com/brainglobe/brainglobe-atlasapi/issues/951. Pinning zarr>=3 above
# fixes it. If you still hit that error after this cell, RESTART THE KERNEL (Kernel > Restart)
# so the newly installed zarr actually gets imported, then re-run from the top.

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [197]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations

%matplotlib inline

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "pdf.fonttype": 42,     # keeps text editable in Illustrator/Inkscape
    "svg.fonttype": "none",
})

## 3. Settings

Edit these to match your data. `ACRONYM` maps your region names to Allen CCFv3 acronyms —
add more pairs if your data has more regions than the 5 listed. `SECTIONS` controls which
coronal level each region is drawn on; check these bregma coordinates against your own
histology before trusting them.

In [ ]:
# ---- file paths ----
RAW_XLSX = "IHC_effect_size_analysis.xlsx"     # your uploaded file, in the same folder as this notebook
RAW_SHEET = "Raw IHC extracted"                 # sheet with one row per animal
LETTERS_CSV = "zif268_means_letters.csv"        # intermediate output, written by this notebook
OUTDIR = "figures_atlas"                        # where the final figures are saved

# ---- atlas ----
ATLAS = "allen_mouse_25um"        # mouse. A rat atlas (e.g. "whs_sd_rat_39um") needs a different BREGMA_AP
BREGMA_AP = 5400                  # um; approx. bregma position along the AP axis in this atlas's coordinate frame

# ---- groups / conditions ----
CONDITION_ORDER = ["SFPT", "ST", "FT"]          # order used for the ANOVA/Tukey and figure order
GROUP_MAP = {"GHAL": "GH AL", "GHFD": "GH FD", "SHAL": "SH AL", "SHFD": "SH FD"}  # raw label -> display label
GROUPS = ["GH FD", "SH AL", "SH FD", "GH AL"]            # which groups get a column in the figure (add "GH AL" for 4)

# ---- plotting ----
CMAP = "magma_r"                  # sequential, colorblind-safe, reads fine in grayscale
VMIN, VMAX = 0, 60                # same scale in all 3 figures so they're directly comparable

# your region names -> Allen CCFv3 acronyms
ACRONYM = {"BNST": "BST", "PVN": "PVH", "SON": "SO", "CeA": "CEA", "BLA": "BLA"}

# (row label, bregma mm, regions shown on that section)
SECTIONS = [
    ("BNST", +0.1, ["BNST"]),
    ("PVN + SON", -0.37, ["PVN", "SON"]),
    ("CeA + BLA", -1.7, ["CeA", "BLA"]),
]

## 4. Stats: one-way ANOVA + Tukey HSD + compact letter display

For each Region x Treatment group: ANOVA across SFPT/ST/FT, then Tukey HSD, then letters.
Two conditions share a letter only if they are *not* significantly different — so no
condition is singled out as a reference. Letters are only produced when the ANOVA itself
was significant; otherwise nothing is drawn for that region/group later on.

In [ ]:
def compact_letters(levels, nonsig_pairs):
    """levels: ordered list, best (highest mean) first.
    nonsig_pairs: set of frozenset({a,b}) pairs that are NOT significantly different.
    Returns dict level -> letter string (e.g. 'a', 'ab', 'b')."""
    n = len(levels)

    def is_clique(group):
        return all(frozenset((a, b)) in nonsig_pairs or a == b
                   for a, b in combinations(group, 2))

    all_subsets = []
    for r in range(n, 0, -1):
        all_subsets += [frozenset(c) for c in combinations(levels, r) if is_clique(c)]
    all_subsets = sorted(set(all_subsets), key=lambda s: (-len(s), sorted(s)))

    needed = set(frozenset((a, b)) for a, b in combinations(levels, 2) if frozenset((a, b)) in nonsig_pairs)
    needed |= set(frozenset((lv,)) for lv in levels)
    remaining = set(needed)

    chosen = []
    for subset in all_subsets:
        subset_needs = set(frozenset((a, b)) for a, b in combinations(subset, 2)) | set(frozenset((lv,)) for lv in subset)
        if subset_needs & remaining:
            chosen.append(subset)
            remaining -= subset_needs
        if not remaining:
            break

    letters = {lv: "" for lv in levels}
    for i, subset in enumerate(chosen):
        letter = chr(ord("a") + i)
        for lv in subset:
            letters[lv] += letter
    return letters


def run_stats(raw_path, raw_sheet, out_path):
    df = pd.read_excel(raw_path, sheet_name=raw_sheet)
    rows, skipped = [], []
    for (region, treat), sub in df.groupby(["Region", "Treatment"]):
        groups = {c: sub.loc[sub.Metric == c, "Value"].dropna().values for c in CONDITION_ORDER}
        if any(len(v) < 2 for v in groups.values()):
            skipped.append((region, treat, {c: len(v) for c, v in groups.items()}))
            continue

        f_stat, p_anova = stats.f_oneway(*[groups[c] for c in CONDITION_ORDER])

        vals = np.concatenate([groups[c] for c in CONDITION_ORDER])
        labs = np.concatenate([[c] * len(groups[c]) for c in CONDITION_ORDER])
        tk = pairwise_tukeyhsd(vals, labs, alpha=0.05)
        tk_df = pd.DataFrame(data=tk._results_table.data[1:], columns=tk._results_table.data[0])

        nonsig_pairs = set()
        for _, r in tk_df.iterrows():
            key = frozenset((r["group1"], r["group2"]))
            if not r["reject"]:
                nonsig_pairs.add(key)

        means = {c: groups[c].mean() for c in CONDITION_ORDER}
        order = sorted(CONDITION_ORDER, key=lambda c: -means[c])
        letters = compact_letters(order, nonsig_pairs)

        for c in CONDITION_ORDER:
            rows.append({
                "condition": c,
                "group": GROUP_MAP.get(treat, treat),
                "region": region,
                "mean": round(means[c], 2),
                "sd": round(groups[c].std(ddof=1), 2),
                "n": len(groups[c]),
                "letter": letters[c],
                "anova_p": round(p_anova, 4),
                "anova_sig": int(p_anova < 0.05),
            })

    out = pd.DataFrame(rows).sort_values(["region", "group", "condition"])
    out.to_csv(out_path, index=False)
    print(f"wrote {out_path}  ({len(out)} rows, {out.region.nunique()} regions, {out.group.nunique()} groups)")
    if skipped:
        print(f"skipped {len(skipped)} region/group combos with <2 animals in a condition:")
        for region, treat, counts in skipped:
            print(f"  {region} / {treat}: n = {counts}")
    return out


letters_df = run_stats(RAW_XLSX, RAW_SHEET, LETTERS_CSV)
letters_df.head(12)

In [ ]:
print(letters_df[letters_df.region == "SON"])
print("SON rows:", (letters_df.region == "SON").sum())

## 5. Download/load the atlas

First run here triggers the Allen atlas download if it isn't cached yet — this is the
step that needs internet and can take a few minutes.

**If this raises `GroupNotFoundError: group not found at path ''`:** your environment has
an old `zarr` (v2.x) shadowing the v3 install from the setup cell — this is a known
brainglobe-atlasapi issue (https://github.com/brainglobe/brainglobe-atlasapi/issues/951).
Run the diagnostic cell below to check, then **restart the kernel** (Kernel > Restart) and
run all cells again from the top -- a `pip install` alone does not swap out an
already-imported module.

In [201]:
import zarr
print("zarr version:", zarr.__version__)
assert int(zarr.__version__.split(".")[0]) >= 3, (
    "zarr is still v2.x in this kernel -- restart the kernel now (Kernel > Restart), "
    "then re-run all cells from the top so the v3 install actually gets picked up."
)

zarr version: 3.1.6


In [ ]:
from brainglobe_atlasapi import BrainGlobeAtlas
from brainglobe_heatmap import Heatmap

atlas = BrainGlobeAtlas(ATLAS)
print(atlas.atlas_name, atlas.resolution)

## 6. Figure-drawing functions

In [203]:
from matplotlib.transforms import Bbox

LINEWIDTH = 0.4          # region outline thickness
INSET_PAD = 0.25         # SON zoom: smaller = tighter crop
INSET_BOX = [0.225, 0.5, 0.55, 0.40]   # SON inset [x, y, w, h] in panel fraction


def draw_panel(ax, fig, values, ap, regs):
    """One coronal panel; adds a zoomed SON inset if SON is on this section."""
    Heatmap(values, position=ap, orientation="frontal", hemisphere="both",
            title="", cmap=CMAP, vmin=VMIN, vmax=VMAX, format="2D", atlas_name=ATLAS
            ).plot_subplot(fig=fig, ax=ax, show_cbar=False, hide_axes=True)
    for p in ax.patches:
        p.set_linewidth(LINEWIDTH)

    son_key = ACRONYM["SON"]
    if "SON" in regs and son_key in values:
        ins = ax.inset_axes(INSET_BOX)
        Heatmap({son_key: values[son_key]}, position=ap, orientation="frontal",
                hemisphere="both", title="", cmap=CMAP, vmin=VMIN, vmax=VMAX,
                format="2D", atlas_name=ATLAS
                ).plot_subplot(fig=fig, ax=ins, show_cbar=False, hide_axes=True)

        # drop the largest patch (gray brain outline), zoom to what's left
        patches = sorted(ins.patches,
                         key=lambda p: -(p.get_path().get_extents().width *
                                         p.get_path().get_extents().height))[1:]
        for p in patches:
            p.set_linewidth(LINEWIDTH)
        if patches:
            box = Bbox.union([p.get_path().get_extents() for p in patches])
            pad = INSET_PAD * max(box.width, box.height)
            ins.set_xlim(box.x0 - pad, box.x1 + pad)
            lo, hi = box.y0 - pad, box.y1 + pad
            ins.set_ylim((hi, lo) if ins.yaxis_inverted() else (lo, hi))
        for s in ins.spines.values():
            s.set_visible(True)
            s.set_linewidth(0.5)


def make_figure(df, group):
    n_rows, n_cols = len(SECTIONS), len(CONDITION_ORDER)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(2.6 * n_cols + 0.6, 2.4 * n_rows + 0.9),
                             squeeze=False)

    for i, (label, breg, regs) in enumerate(SECTIONS):
        ap = BREGMA_AP - breg * 1000
        for j, cond in enumerate(CONDITION_ORDER):
            ax = axes[i][j]
            sub = df[(df.group == group) & (df.condition == cond) & (df.region.isin(regs))]
            values = {ACRONYM[r]: v for r, v in zip(sub.region, sub["mean"])}
            draw_panel(ax, fig, values, ap, regs)

            if i == 0:
                ax.set_title(cond, fontsize=9, fontweight="bold")
            if j == 0:
                ax.annotate(f"{label}\n(~Bregma {breg:+.1f} mm)", xy=(0, 0.5),
                            xycoords="axes fraction", xytext=(-6, 0),
                            textcoords="offset points", ha="right", va="center",
                            rotation=90, fontsize=8)

    fig.suptitle(group, fontsize=11, fontweight="bold")
    fig.subplots_adjust(left=0.10, right=0.98, top=0.90, bottom=0.14, wspace=0.02, hspace=0.05)
    cax = fig.add_axes([0.35, 0.06, 0.4, 0.02])
    cb = fig.colorbar(ScalarMappable(norm=Normalize(VMIN, VMAX), cmap=CMAP),
                      cax=cax, orientation="horizontal")
    cb.set_label("Mean % Zif268+ cells", fontsize=8)
    cb.outline.set_linewidth(0.4)
    return fig

## 7. Generate the three figures

Each one displays inline below and is also saved to `figures_atlas/` as PDF, SVG and PNG.

In [ ]:
import os
os.makedirs(OUTDIR, exist_ok=True)

figs = {}
for group in GROUPS:
    fig = make_figure(letters_df, group)
    tag = group.replace(" ", "")
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUTDIR}/zif268_atlas_{tag}.{ext}", dpi=300, bbox_inches="tight")
    figs[group] = fig
    plt.show()
    print("saved", group)

## Notes / things to double check

- **Bregma coordinates** (`SECTIONS`) are approximate — verify against your own histology.
- **Acronym mapping** (`ACRONYM`) uses Allen parent structures; if your IHC targeted a
  specific subnucleus, look up a more precise acronym via `atlas.structures`.
- **Species**: currently set to mouse (`allen_mouse_25um`). For rat or another species,
  change `ATLAS` and re-derive `BREGMA_AP`.
- A region/group combo with fewer than 2 animals in any condition is skipped automatically
  (printed above) rather than crashing the ANOVA.
- Multiple-comparisons correction: this uses raw p < 0.05 for both the ANOVA screen and
  Tukey HSD. With many region x group combinations tested, consider whether you want an
  FDR correction across regions — ask if you'd like that added.
- **`GroupNotFoundError` when loading the atlas** — a known brainglobe-atlasapi bug caused
  by an old `zarr` v2.x already present in your environment
  (https://github.com/brainglobe/brainglobe-atlasapi/issues/951). Fixed by the `zarr>=3`
  pin in the install cell, but you must **restart the kernel** afterward for it to take
  effect, since `pip install` doesn't retroactively change an already-imported module.